In [1]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from locations import extract_url_map
from events import parse_mmd_taxonomy, extract_events
from timespan import parse_timespan

In [2]:
root_path = "../data/schede mappatura/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
}

In [3]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede

['../data/schede mappatura/bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx',
 '../data/schede mappatura/david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopoi_charlotte_bruenn_IS_S_00027 (bozza).xlsx',
 '../data/schede mappatura/0_template/chronotopoi_template.xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v3.xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx']

## A list of individual sources for experimentation

ignored in the oveall logic

In [4]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

['../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx']

After identification of all sources

# Shortlist processable sources

In [5]:
overview = {
    "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [6]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = [
            "event_label",
            "event_type",
            "place_name",
            "place_type",
            "place_category",
            "wikidata_qid",
            "geonames_id",
            "google maps",
            "date_certainty",
            "date_label",
            "memorial_inscription",
            "source_doc",
            "source_timecode",
            "source_quote",
            "external_links",
            "notes",
        ]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        # df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)
df

stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx Josef Stern Index(['event_label', 'event_type', 'place_name', 'place_type',
       'place_category', 'wikidata_qid', 'geonames_id', 'google maps',
       'date_certainty', 'date_label', 'memorial_inscription', 'source_doc',
       'source_timecode', 'source_quote', 'external_links', 'notes'],
      dtype='str')


,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,memorial_inscription,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern


# Locations

In [7]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip()
        )

    locs[name].update(urls[0])
print(locs)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:00<00:00, 16815.94it/s]

{'Gießen, Marktplatz 11': {'label': 'ort_der_zeit', 'www.geonames.org': 'https://www.geonames.org/2920512.0'}, 'Gießen': {'label': 'reise_zurueck', 'www.geonames.org': 'https://www.geonames.org/2920512/giessen.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q564579'}, 'Löberstraße 20': {'label': 'alte_heimat', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Ghettohaus, Walltorstraße 48': {'label': 'alte_heimat,Ghettohaus', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Schlesien': {'label': 'alte_heimat'}, 'Berlin': {'label': 'alte_heimat', 'www.geonames.org': 'https://www.geonames.org/2950159/berlin.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q64'}, 'Synagoge am Börneplatz, Frankfurt/M': {'label': 'alte_heimat,Synagoge', 'www.geonames.org': 'https://www.geonames.org/6553153/frankfurt-am-main.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q

## Update preexisting locations

In [8]:
rich = pd.read_excel("locations.xlsx", dtype=str)
rich = rich.set_index(["location"])

for location, row in locs.items():
    if location not in rich.index:
        rich.loc[location] = locs[location]

    # Add any new columns from locs that don't exist yet
    for col, val in row.items() if isinstance(row, dict) else {}:
        if col not in rich.columns:
            rich[col] = ""
        rich.loc[location, col] = str(val)

rich.to_excel("locations.xlsx")

# Events

TODO: incomplete due to too much noise. Issues:

- use LL or Lebenslauf, currently extracted as one, but need to be two equivalent
- "alter heimant" instread of "alte heimat"
- "Transport", "Tod des Vaters",  are not label

In [9]:
event_taxonomy = parse_mmd_taxonomy("../model/tassonomia.mmd")
events = sorted(set(event_taxonomy.keys()), key=lambda x: -len(x))
len(events), events[:5] + ["..."] + events[-5:]

(340,
 ['Città/regioni tedesche, austriache, ceche',
  'SR – Spazi sociali / istituzioni',
  'Friedrichswerdersche Gymnasium',
  'Europa – altri paesi e luoghi',
  'Berlino – quartieri e luoghi',
  '...',
  'Zug',
  'SPD',
  'USA',
  'KPD',
  'Tod'])

In [10]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,memorial_inscription,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name,event
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"[alte Heimat, Geburt]"
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"[alte Heimat, Grundschule]"
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,"[Realgymnasium, alte Heimat]"
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,[Wohnort]
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,[Abgang von der Schule]
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,[Wohnort]
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,[Hachschara]
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,"[Verwandte, Bei n]"
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,[Jeschiwa]
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"[Abschied von den Eltern, Abfahrt, >]"


# Timespan

In [11]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,...,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name,event,time_start,time_end
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,...,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"[alte Heimat, Geburt]",1921-06-15,1921-06-15
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,...,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"[alte Heimat, Grundschule]",1928-01-01,1932-12-31
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,...,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,"[Realgymnasium, alte Heimat]",1932-01-01,1932-12-31
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,...,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,[Wohnort],1933-01-01,1933-12-31
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,...,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,[Abgang von der Schule],1935-01-01,1935-12-31
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,...,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,[Wohnort],1935-01-01,1935-12-31
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,...,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,[Hachschara],1935-01-01,1935-12-31
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,...,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,"[Verwandte, Bei n]",1935-01-01,1935-12-31
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,...,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,[Jeschiwa],1936-01-01,1936-12-31
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,...,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"[Abschied von den Eltern, Abfahrt, >]",1936-01-01,1936-12-31


# Notes

left unprocessed for now

In [12]:
set(df["notes"])

{'11974166.0 https://www.wikidata.org/wiki/Q116016915\nhttps://www.geonames.org/11974166/grossen-linden.html',
 '2888549.0 https://www.wikidata.org/wiki/Q1571834\nhttps://www.geonames.org/2888549/klein-linden.html',
 '2891951.0 probable 1936 https://www.wikidata.org/wiki/Q15979\nhttps://www.geonames.org/2891951/kehl.html',
 '2920512.0 certain 15/06/1921 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579\nhttps://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 '2920512.0 probable 1975 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579',
 '293304.0 uncertain 1940-1944 https://www.wikidata.org/wiki/Q550025\nhttps://www.geonames.org/293304/tirat-tsvi.html',
 '2950159.0 probable 1935 31min 31s https://www.wikidata.org/wiki/Q64\nhttps://www.geonames.org/2950159/berlin.html',
 '2995469.0 probable 1936 https://www.wikidata.org/wiki/Q23482\nhttps://www.geonames.org/2995469/marseille.html',
 '31min 58

# Links

left unprocessed for now

In [16]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

{'https://www.geonames.org/11974166/grossen-linden.html',
 'https://www.geonames.org/2888549/klein-linden.html',
 'https://www.geonames.org/2891951/kehl.html',
 'https://www.geonames.org/2920512/giessen.html',
 'https://www.geonames.org/293165/jezreel-valley.html',
 'https://www.geonames.org/293304/tirat-tsvi.html',
 'https://www.geonames.org/294801/haifa.html',
 'https://www.geonames.org/2950159/berlin.html',
 'https://www.geonames.org/295211/-en-hanaziv.html',
 'https://www.geonames.org/2995469/marseille.html',
 'https://www.geonames.org/6290300/frankfurt-hauptbahnhof.html',
 'https://www.geonames.org/6553153/frankfurt-am-main.html',
 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 'https://www.wikidata.org/wiki/Q111620976',
 'https://www.wikidata.org/wiki/Q116016915',
 'https://www.wikidata.org/wiki/Q1375288',
 'https://www.wikidata.org/wiki/Q1571834',
 'https://www.wikidata.org/wiki/Q15979',
 'https://www.wikidata.org/wiki/Q165368',
 'https://ww